# 02d — Nested CV Optuna Tuning (Winning Configuration Only)

**Stage B** of the revised pipeline. Runs Optuna hyperparameter search on the single
(model, window, horizon, covariate-group) configuration selected by
`02c_model_selection_cv.ipynb` (Stage A) -- not averaged across scenarios, which is
what reviewer feedback flagged as an "unusual" objective in the old
`legacy_pre_revision/01_hyperparameter_tuning.ipynb`.

**Nested CV**: for each of the same 5 outer folds Stage A used for this configuration,
an inner Optuna search (bounded strictly to that outer fold's own training range) picks
hyperparameters, which are then evaluated once, out-of-sample, on the outer fold's test
window. Optuna's objective never sees outer test data -- `cv_lib.make_inner_cv_objective`
asserts this. A final production tuning pass (same inner-CV mechanism, over all data up
to the last outer fold's training cutoff) produces the single hyperparameter set used
downstream by `03_shap_analysis.ipynb`.

## Output
- `nested_cv_outer_fold_results.csv` -- per outer fold: default MAPE (from Stage A) vs. tuned MAPE, DA, MASE, regime flags
- `nested_cv_default_vs_tuned_summary.csv` -- aggregated improvement %
- `saved_models/optuna_tuning_results.joblib` -- final production hyperparameters, same schema as the old `01` notebook's output


In [1]:
import glob
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler

import cv_lib as cv

optuna.logging.set_verbosity(optuna.logging.WARNING)

df_merged = joblib.load(sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)[0])
winning_config = joblib.load("saved_models/winning_config.joblib")
df_stage_a = pd.read_csv("stage_a_fold_results.csv")

print("Winning configuration from Stage A:")
for k, v in winning_config.items():
    print(f"  {k}: {v}")

MODEL_NAME = winning_config["model"]
COV_VARS = winning_config["cov_vars"]
WINDOW = winning_config["window"]
HORIZON = winning_config["horizon"]
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
EMBARGO = HORIZON
N_TRIALS_PER_OUTER_FOLD = 40
N_TRIALS_PRODUCTION = 50


Winning configuration from Stage A:
  model: ExtraTrees
  covariates: Screening1
  cov_vars: ['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio']
  window: 120
  horizon: 1
  default_mape_mean: 0.6359999999999999
  default_mape_std: 0.15013808644044985


## Rebuild outer folds -- must match Stage A exactly

Stage A used `expanding_window_folds(n, n_folds=5, embargo=horizon)` on this exact
configuration's series. Rebuilding it here with the same parameters and asserting the
fold date ranges match `stage_a_fold_results.csv` is what makes the default-vs-tuned
comparison a genuine paired comparison on identical test windows, not just "similar."


In [2]:
target_ts, cov_ts = cv.to_series(df_merged, "IHSG", COV_VARS if COV_VARS else None)
n = len(target_ts)
outer_folds = cv.expanding_window_folds(n, n_folds=N_OUTER_FOLDS, embargo=EMBARGO)
print(f"n={n}, {len(outer_folds)} outer folds")

df_winner_default = df_stage_a[
    (df_stage_a["Model"] == MODEL_NAME) &
    (df_stage_a["Covariates"] == winning_config["covariates"]) &
    (df_stage_a["Window"] == WINDOW) &
    (df_stage_a["Horizon"] == HORIZON)
].sort_values("Fold").reset_index(drop=True)

assert len(df_winner_default) == len(outer_folds), (
    f"Stage A has {len(df_winner_default)} fold rows for the winning config, "
    f"rebuilt {len(outer_folds)} outer folds here -- CV parameters must have drifted"
)
for f, (_, row) in zip(outer_folds, df_winner_default.iterrows()):
    rebuilt_test_start = str(target_ts[f["test_start"]].start_time().date())
    assert rebuilt_test_start == row["test_start_date"], (
        f"fold {f['fold']} date mismatch: rebuilt {rebuilt_test_start} vs "
        f"Stage A {row['test_start_date']} -- not a valid paired comparison"
    )
print("Outer folds match Stage A's fold boundaries for this configuration -- paired comparison is valid.")


n=2595, 5 outer folds
Outer folds match Stage A's fold boundaries for this configuration -- paired comparison is valid.


## Nested CV: per outer fold, inner-CV-tuned hyperparameters evaluated out-of-sample

For each outer fold `k`, `cv_lib.make_inner_cv_objective` builds an Optuna objective
bounded strictly to `target_ts[:outer_train_end_k]` (asserted). The best inner-CV
hyperparameters are then evaluated exactly once, out-of-sample, on
`[outer_test_start_k:outer_test_end_k]` via `cv.run_fold` -- that single evaluation is
what's reported for fold `k`.


In [3]:
import os

# Resume-aware: checkpoint after each outer fold's Optuna study + evaluation,
# since this environment has shown it can interrupt long-running background
# processes -- an outer fold with N_TRIALS_PER_OUTER_FOLD=40 x 3 inner folds
# is a meaningful chunk of work to lose.
if os.path.exists("nested_cv_outer_fold_results.csv"):
    df_resume = pd.read_csv("nested_cv_outer_fold_results.csv")
    outer_results = df_resume.to_dict("records")
    done_folds = set(df_resume["Fold"])
    print(f"Resuming: {len(outer_results)} outer folds already done: {sorted(done_folds)}")
else:
    outer_results = []
    done_folds = set()

for f in outer_folds:
    if f["fold"] in done_folds:
        print(f"Outer fold {f['fold']} ... skipped (already in checkpoint)")
        continue

    print(f"\n--- Outer fold {f['fold']} (train_end={f['train_end']}, "
          f"test=[{f['test_start']}:{f['test_end']}]) ---")

    objective, inner_folds = cv.make_inner_cv_objective(
        MODEL_NAME, target_ts, cov_ts, WINDOW, HORIZON,
        outer_train_end=f["train_end"], n_inner_folds=N_INNER_FOLDS, embargo=EMBARGO,
    )
    print(f"  Inner folds: {len(inner_folds)} (all bounded by outer_train_end={f['train_end']})")

    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42),
                                 study_name=f"{MODEL_NAME}_outer{f['fold']}")
    study.optimize(objective, n_trials=N_TRIALS_PER_OUTER_FOLD, show_progress_bar=False)
    print(f"  Best inner-CV MAPE: {study.best_value:.4f}  params={study.best_params}")

    tuned_metrics = cv.run_fold(MODEL_NAME, study.best_params, target_ts, cov_ts, WINDOW, HORIZON,
                                 f["train_end"], f["test_start"], f["test_end"])
    default_row = df_winner_default[df_winner_default["Fold"] == f["fold"]].iloc[0]

    outer_results.append({
        "Fold": f["fold"],
        "train_end_date": tuned_metrics["train_end_date"],
        "test_start_date": tuned_metrics["test_start_date"],
        "test_end_date": tuned_metrics["test_end_date"],
        "regime_flags": tuned_metrics["regime_flags"],
        "default_mape": default_row["mape"], "tuned_mape": tuned_metrics["mape"],
        "default_da": default_row["da"], "tuned_da": tuned_metrics["da"],
        "default_mase": default_row["mase"], "tuned_mase": tuned_metrics["mase"],
        "best_params": study.best_params,
    })
    improvement = (default_row["mape"] - tuned_metrics["mape"]) / default_row["mape"] * 100
    print(f"  Outer test: default MAPE={default_row['mape']:.4f}%  tuned MAPE={tuned_metrics['mape']:.4f}%  "
          f"improvement={improvement:.2f}%"
          + (f"  [REGIME: {tuned_metrics['regime_flags']}]" if tuned_metrics["regime_flags"] else ""))

    pd.DataFrame(outer_results).to_csv("nested_cv_outer_fold_results.csv", index=False)
    print(f"  -> Checkpoint saved ({len(outer_results)} outer folds)")

df_outer = pd.DataFrame(outer_results)
df_outer.to_csv("nested_cv_outer_fold_results.csv", index=False)
print(f"\nSaved: nested_cv_outer_fold_results.csv ({len(df_outer)} rows)")


--- Outer fold 0 (train_end=1038, test=[1039:1428]) ---
  Inner folds: 3 (all bounded by outer_train_end=1038)


  Best inner-CV MAPE: 0.5218  params={'n_estimators': 900, 'max_depth': 7, 'max_features': 0.3, 'min_samples_split': 13, 'min_samples_leaf': 6}


  Outer test: default MAPE=0.8084%  tuned MAPE=0.8077%  improvement=0.09%  [REGIME: covid_crash]
  -> Checkpoint saved (1 outer folds)

--- Outer fold 1 (train_end=1329, test=[1330:1719]) ---
  Inner folds: 3 (all bounded by outer_train_end=1329)


  Best inner-CV MAPE: 0.6229  params={'n_estimators': 900, 'max_depth': 12, 'max_features': 'log2', 'min_samples_split': 8, 'min_samples_leaf': 8}


  Outer test: default MAPE=0.7826%  tuned MAPE=0.7788%  improvement=0.49%  [REGIME: covid_crash]
  -> Checkpoint saved (2 outer folds)

--- Outer fold 2 (train_end=1620, test=[1621:2010]) ---
  Inner folds: 3 (all bounded by outer_train_end=1620)


...

  Outer test: default MAPE=0.5697%  tuned MAPE=0.5711%  improvement=-0.25%  [REGIME: rate_hike_cycle_2022]
  -> Checkpoint saved (3 outer folds)

--- Outer fold 3 (train_end=1911, test=[1912:2301]) ---
  Inner folds: 3 (all bounded by outer_train_end=1911)


  Best inner-CV MAPE: 0.7421  params={'n_estimators': 200, 'max_depth': 15, 'max_features': 'log2', 'min_samples_split': 15, 'min_samples_leaf': 5}


  Outer test: default MAPE=0.4734%  tuned MAPE=0.4749%  improvement=-0.32%  [REGIME: rate_hike_cycle_2022]
  -> Checkpoint saved (4 outer folds)

--- Outer fold 4 (train_end=2202, test=[2203:2592]) ---
  Inner folds: 3 (all bounded by outer_train_end=2202)


  Best inner-CV MAPE: 0.6132  params={'n_estimators': 900, 'max_depth': 12, 'max_features': 0.7, 'min_samples_split': 3, 'min_samples_leaf': 10}


  Outer test: default MAPE=0.5459%  tuned MAPE=0.5441%  improvement=0.33%
  -> Checkpoint saved (5 outer folds)

Saved: nested_cv_outer_fold_results.csv (5 rows)


## Summary: default vs. tuned, per fold and aggregated

Flags any fold overlapping a known regime shift (2020 COVID crash, 2022 rate-hike
cycle) for separate discussion rather than folding it silently into the average.


In [4]:
df_outer["improvement_pct"] = (df_outer["default_mape"] - df_outer["tuned_mape"]) / df_outer["default_mape"] * 100

summary = {
    "model": MODEL_NAME, "covariates": winning_config["covariates"],
    "window": WINDOW, "horizon": HORIZON,
    "default_mape_mean": df_outer["default_mape"].mean(), "default_mape_std": df_outer["default_mape"].std(),
    "tuned_mape_mean": df_outer["tuned_mape"].mean(), "tuned_mape_std": df_outer["tuned_mape"].std(),
    "mean_improvement_pct": df_outer["improvement_pct"].mean(),
    "n_outer_folds": len(df_outer),
    "n_folds_with_regime_flag": df_outer["regime_flags"].notna().sum(),
}
pd.DataFrame([summary]).to_csv("nested_cv_default_vs_tuned_summary.csv", index=False)

print("Default vs. tuned MAPE, per outer fold:")
display(df_outer[["Fold", "test_start_date", "test_end_date", "default_mape", "tuned_mape",
                   "improvement_pct", "regime_flags"]])

flagged = df_outer[df_outer["regime_flags"].notna()]
if len(flagged):
    print(f"\n{len(flagged)}/{len(df_outer)} folds overlap a flagged regime window -- "
          "discuss these separately rather than averaging them in silently:")
    display(flagged[["Fold", "test_start_date", "test_end_date", "regime_flags", "tuned_mape"]])

print(f"\nMean improvement (default -> tuned): {summary['mean_improvement_pct']:.2f}%")
print(f"Saved: nested_cv_default_vs_tuned_summary.csv")


Default vs. tuned MAPE, per outer fold:


,Fold,test_start_date,test_end_date,default_mape,tuned_mape,improvement_pct,regime_flags
0,0,2019-02-15,2020-08-12,0.8084,0.8077,0.086591,covid_crash
1,1,2020-03-30,2021-09-23,0.7826,0.7788,0.485561,covid_crash
2,2,2021-05-11,2022-11-04,0.5697,0.5711,-0.245743,rate_hike_cycle_2022
3,3,2022-06-22,2023-12-18,0.4734,0.4749,-0.316857,rate_hike_cycle_2022
4,4,2023-08-03,2025-01-28,0.5459,0.5441,0.329731,NaN



4/5 folds overlap a flagged regime window -- discuss these separately rather than averaging them in silently:


,Fold,test_start_date,test_end_date,regime_flags,tuned_mape
0,0,2019-02-15,2020-08-12,covid_crash,0.8077
1,1,2020-03-30,2021-09-23,covid_crash,0.7788
2,2,2021-05-11,2022-11-04,rate_hike_cycle_2022,0.5711
3,3,2022-06-22,2023-12-18,rate_hike_cycle_2022,0.4749



Mean improvement (default -> tuned): 0.07%
Saved: nested_cv_default_vs_tuned_summary.csv


## Final production tuning pass

One more Optuna study, using the same inner-CV mechanism, bounded by the *last* outer
fold's training cutoff (i.e. all data up to but not including the final held-out test
window) -- produces the single hyperparameter set used downstream for SHAP.


In [5]:
final_train_end = outer_folds[-1]["train_end"]
prod_objective, prod_inner_folds = cv.make_inner_cv_objective(
    MODEL_NAME, target_ts, cov_ts, WINDOW, HORIZON,
    outer_train_end=final_train_end, n_inner_folds=N_INNER_FOLDS, embargo=EMBARGO,
)
print(f"Production tuning: {len(prod_inner_folds)} inner folds bounded by train_end={final_train_end}")

prod_study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42), study_name=f"{MODEL_NAME}_production")
prod_study.optimize(prod_objective, n_trials=N_TRIALS_PRODUCTION, show_progress_bar=False)

print(f"Production best inner-CV MAPE: {prod_study.best_value:.4f}")
print(f"Production best params: {prod_study.best_params}")

# Same schema as the old 01_hyperparameter_tuning.ipynb output, but containing
# only the single winning model -- 03_shap_analysis.ipynb's TUNING[model]['best_params']
# lookup needs no code changes for this load line.
TUNING_RESULTS = {
    MODEL_NAME: {"best_params": prod_study.best_params, "best_value": prod_study.best_value},
}
joblib.dump(TUNING_RESULTS, "saved_models/optuna_tuning_results.joblib")
print("Saved: saved_models/optuna_tuning_results.joblib")

# Also persist which config this is, for 03/04's benefit.
joblib.dump({**winning_config, "production_train_end": final_train_end,
             "production_test_start": outer_folds[-1]["test_start"],
             "production_test_end": outer_folds[-1]["test_end"]},
            "saved_models/final_config.joblib")
print("Saved: saved_models/final_config.joblib")


Production tuning: 3 inner folds bounded by train_end=2202


Production best inner-CV MAPE: 0.6129
Production best params: {'n_estimators': 800, 'max_depth': 12, 'max_features': 0.7, 'min_samples_split': 3, 'min_samples_leaf': 10}
Saved: saved_models/optuna_tuning_results.joblib
Saved: saved_models/final_config.joblib
